# Dissertation Benchmark Generation and Validation (v3)

This notebook is for the **final F1--F4 benchmark**, not the older occasion-based demonstration set.

It does four things:

1. calls `generate_dissertation_scenarios.py`;
2. generates the 12 experimental scenarios for seeds 1--10 (120 instances);
3. validates structural integrity and the intended F1--F4 paired-family design; and
4. freezes the accepted JSON files with CSV/JSON manifests and SHA-256 hashes.

Keep this notebook in the same folder as:

```text
generate_dissertation_scenarios.py
```

The generator's internal `scenario.validate()` and `design_evidence` are construction checks.  
They **do not replace the exact MILP feasibility test**. An optional hook for the exact MILP is provided near the end.

**v2 fix:** aligned `GeneratorConfig` with the final F1--F4 generator. Asset counts are controlled by `SYSTEM_SCALE_PROFILES`, not a `shiploader_count` constructor argument.


**v3 fix:** aligned the F4 family validator with the final calibrated treatment: F4-L has capacity conflict only; F4-M and F4-H have the same hard X1 maintenance-incompatibility exposure, while F4-H is made more severe through stronger activity capacity reductions.


## 1. Import the current generator and verify the interface

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import hashlib
import json
import math
import statistics

import generate_dissertation_scenarios as gen

required_names = {
    "F1_system_scale_L", "F1_system_scale_M", "F1_system_scale_H",
    "F2_mode_tradeoff_L", "F2_mode_tradeoff_M", "F2_mode_tradeoff_H",
    "F3_yard_flow_L", "F3_yard_flow_M", "F3_yard_flow_H",
    "F4_operational_conflict_L", "F4_operational_conflict_M",
    "F4_operational_conflict_H",
}

experimental_names = tuple(gen.EXPERIMENTAL_SCENARIO_NAMES)
missing = required_names - set(experimental_names)

assert not missing, (
    "This notebook expects the final F1--F4 generator. "
    f"Missing scenario names: {sorted(missing)}"
)

print("Generator:", gen.__file__)
print("Randomization version:", getattr(gen, "GENERATOR_RANDOMIZATION_VERSION", "unknown"))
print("RNG stream version:", getattr(gen, "RNG_STREAM_VERSION", "unknown"))
print("Experimental scenarios:", len(experimental_names))
for name in experimental_names:
    print(" -", name)

## 2. Final benchmark settings

In [ ]:
SEEDS = tuple(range(1, 11))       # final dissertation sample: 10 seeds
OUTPUT_DIR = Path("generated_final_benchmark_new")
WRITE_JSON = True

# Do not expose the construction witness to the optimisation experiment.
INCLUDE_REFERENCE_SCHEDULE = False

def make_config(seed: int):
    """
    Build GeneratorConfig using the CURRENT final generator interface.

    Physical asset counts are NOT configured here. They are controlled by
    F1's SYSTEM_SCALE_PROFILES inside generate_dissertation_scenarios.py.
    """
    cfg = gen.GeneratorConfig(
        horizon_days=90,
        seed=int(seed),
        base_supply_tpd=178_000.0,
        base_demand_tpd=184_000.0,
        noise_fraction=0.035,
        number_of_products=4,
        operating_hours_per_day=24.0,
        train_unloader_rate_tph=2917.0,
        stacker_rate_tph=2292.0,
        reclaimer_rate_tph=2708.0,
        shiploader_rate_tph=4583.0,
        arc_secondary_rate_tph=1875.0,
    )
    cfg.validate()
    return cfg

# Show the live constructor fields so an interface mismatch is obvious.
config_fields = tuple(gen.GeneratorConfig.__dataclass_fields__)
print("GeneratorConfig fields:")
for field_name in config_fields:
    print(" -", field_name)

print(
    f"\nPlanned benchmark: {len(experimental_names)} scenarios x {len(SEEDS)} seeds "
    f"= {len(experimental_names) * len(SEEDS)} instances"
)


## 3. Preview one generated instance

In [ ]:
PREVIEW_SCENARIO = "F1_system_scale_M"
PREVIEW_SEED = 1

preview = gen.build_scenario(PREVIEW_SCENARIO, make_config(PREVIEW_SEED))
preview_data = preview.optimizer_objects()

print("Scenario:", preview.metadata["scenario_name"])
print("Family:", preview.metadata.get("scenario_family"))
print("Level:", preview.metadata.get("scenario_level"))
print("System-scale profile:", preview.metadata.get("system_scale_profile"))
print("Seed:", PREVIEW_SEED)
print("Days:", len(preview.days))
print("Assets:", len(preview.assets))
print("Arcs:", len(preview.arcs))
print("Maintenance operations:", len(preview.maintenance_tasks))
print("Priority activities:", len(preview.priority_activities))
print("Modes:", preview_data["K"])
print(
    "Built-in design checks:",
    preview.design_evidence["occasion_checks"]["all_checks_pass"],
)


## 4. Validation helpers

The checks below are deliberately split into:

- **instance checks**: one generated case is internally consistent;
- **family checks**: L/M/H differ only in the intended treatment;
- **seed-diversity checks**: different seeds genuinely produce different realizations.

These are generator/data checks only.

In [ ]:
EXPECTED_PRODUCTS = {"P1", "P2", "P3", "X1"}
EXPECTED_ZONES = {"Z1", "Z2", "Z3", "Z4"}
EXPECTED_MODES = {"full_shutdown", "partial_30", "partial_70"}

def _round_nested(value, digits=8):
    if isinstance(value, dict):
        return tuple(sorted((str(k), _round_nested(v, digits)) for k, v in value.items()))
    if isinstance(value, (list, tuple)):
        return tuple(_round_nested(v, digits) for v in value)
    if isinstance(value, float):
        return round(value, digits)
    return value

def supply_signature(scenario):
    return tuple(
        (r.day, r.product_id, round(r.tonnes, 8))
        for r in scenario.inbound_supply
    )

def demand_signature(scenario):
    return tuple(
        (r.day, r.product_id, round(r.tonnes, 8))
        for r in scenario.outbound_demand
    )

def inventory_signature(scenario):
    return _round_nested(scenario.initial_inventory)

def topology_signature(scenario):
    assets = tuple(sorted(
        (a.asset_id, a.stage, round(a.nominal_rate_tph, 8))
        for a in scenario.assets
    ))
    arcs = tuple(sorted(
        (a.from_node, a.to_node, round(a.capacity_tph, 8), tuple(a.allowed_products))
        for a in scenario.arcs
    ))
    return assets, arcs

def maintenance_core_signature(scenario):
    rows = []
    for task in sorted(scenario.maintenance_tasks, key=lambda x: x.task_id):
        full = next(m for m in task.modes if m.mode_id == "full_shutdown")
        rows.append((
            task.task_id,
            task.asset_id,
            task.earliest_start_day,
            task.latest_start_day,
            task.lifecycle_due_day,
            full.duration_days,
        ))
    return tuple(rows)

def daily_total(records):
    totals = defaultdict(float)
    for r in records:
        totals[r.day] += r.tonnes
    return dict(totals)

def mean_daily_total(records):
    values = list(daily_total(records).values())
    return statistics.fmean(values)

def scenario_level(name):
    return name.rsplit("_", 1)[-1]

def family_prefix(name):
    return name.rsplit("_", 1)[0]

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def check_instance(scenario):
    errors = []

    # Re-run the generator's authoritative structural validator.
    try:
        scenario.validate()
    except Exception as exc:
        errors.append(f"scenario.validate(): {exc}")

    if scenario.days != list(range(90)):
        errors.append("days are not exactly 0..89")

    if {p.product_id for p in scenario.products} != EXPECTED_PRODUCTS:
        errors.append("product set is not {P1,P2,P3,X1}")

    if {z.zone_id for z in scenario.zones} != EXPECTED_ZONES:
        errors.append("yard-zone set is not {Z1,Z2,Z3,Z4}")

    asset_ids = {a.asset_id for a in scenario.assets}
    node_ids = {"RAIL_SOURCE", "VESSEL_SINK", *asset_ids, *EXPECTED_ZONES}

    for arc in scenario.arcs:
        if arc.from_node not in node_ids or arc.to_node not in node_ids:
            errors.append(f"arc {arc.arc_id} has invalid endpoint")
        if arc.capacity_tph <= 0:
            errors.append(f"arc {arc.arc_id} has non-positive capacity")

    for asset in scenario.assets:
        if asset.nominal_rate_tph <= 0:
            errors.append(f"asset {asset.asset_id} has non-positive rate")

    zone_map = {z.zone_id: z for z in scenario.zones}
    for zone_id, by_product in scenario.initial_inventory.items():
        zone = zone_map[zone_id]
        total = sum(by_product.values())
        if total < zone.minimum_inventory_tonnes - 1e-6:
            errors.append(f"{zone_id} initial inventory below minimum")
        if total >= zone.capacity_tonnes + 1e-6:
            errors.append(f"{zone_id} initial inventory at/above capacity")
        for product_id, tonnes in by_product.items():
            if tonnes < -1e-8:
                errors.append(f"{zone_id}/{product_id} has negative initial stock")
            if product_id not in zone.allowed_products and abs(tonnes) > 1e-8:
                errors.append(f"{zone_id}/{product_id} violates compatibility")

    for record in (*scenario.inbound_supply, *scenario.outbound_demand):
        if record.tonnes < -1e-8:
            errors.append(f"negative supply/demand for {record.product_id} day {record.day}")

    for task in scenario.maintenance_tasks:
        if task.asset_id not in asset_ids:
            errors.append(f"{task.task_id} assigned to missing asset {task.asset_id}")
        if not (0 <= task.earliest_start_day <= task.latest_start_day):
            errors.append(f"{task.task_id} has invalid EM/LM")
        modes = {m.mode_id: m for m in task.modes}
        if set(modes) != EXPECTED_MODES:
            errors.append(f"{task.task_id} has mode set {sorted(modes)}")
            continue
        durations = [
            modes["full_shutdown"].duration_days,
            modes["partial_30"].duration_days,
            modes["partial_70"].duration_days,
        ]
        if not (0 < durations[0] < durations[1] < durations[2]):
            errors.append(f"{task.task_id} has non-monotone durations {durations}")
        if task.latest_start_day + max(durations) > len(scenario.days):
            errors.append(f"{task.task_id} longest mode can exceed horizon")

    data = scenario.optimizer_objects()
    if set(data["K"]) != EXPECTED_MODES:
        errors.append(f"optimizer K is {data['K']}")

    # Common X1 route must exist.
    route = [
        ("RAIL_SOURCE", "TU2"),
        ("TU2", "ST3"),
        ("ST3", "Z4"),
        ("Z4", "RC2"),
        ("RC2", "SL1"),
        ("SL1", "VESSEL_SINK"),
    ]
    arc_pairs = {(a.from_node, a.to_node) for a in scenario.arcs}
    for pair in route:
        if pair not in arc_pairs:
            errors.append(f"missing X1 route arc {pair}")

    design_ok = bool(
        scenario.design_evidence.get("occasion_checks", {}).get("all_checks_pass", False)
    )
    if not design_ok:
        errors.append("generator design_evidence reports failed check(s)")

    return errors

## 5. Generate and validate all 120 benchmark instances

In [ ]:
cases = {}
instance_rows = []

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for seed in SEEDS:
    cfg = make_config(seed)

    for name in experimental_names:
        scenario = gen.build_scenario(name, cfg)
        errors = check_instance(scenario)
        cases[(name, seed)] = scenario

        output_path = ""
        sha256 = ""

        if WRITE_JSON:
            case_dir = OUTPUT_DIR / name / f"seed_{seed:03d}"
            case_dir.mkdir(parents=True, exist_ok=True)
            path = case_dir / f"{name}_seed_{seed:03d}.json"
            gen.write_scenario(
                scenario,
                path,
                include_reference_schedule=INCLUDE_REFERENCE_SCHEDULE,
            )
            output_path = str(path)
            sha256 = file_sha256(path)

        md = scenario.metadata
        instance_rows.append({
            "instance_id": f"{name}_seed_{seed:03d}",
            "scenario": name,
            "family": md.get("scenario_family", family_prefix(name)),
            "level": md.get("scenario_level", scenario_level(name)),
            "seed": seed,
            "assets": len(scenario.assets),
            "arcs": len(scenario.arcs),
            "maintenance_operations": len(scenario.maintenance_tasks),
            "activities": len(scenario.priority_activities),
            "effective_capacity_tph": md.get("effective_network_capacity_tph", ""),
            "mean_supply_tpd": mean_daily_total(scenario.inbound_supply),
            "mean_demand_tpd": mean_daily_total(scenario.outbound_demand),
            "structural_pass": len(errors) == 0,
            "structural_errors": " | ".join(errors),
            "json_file": output_path,
            "sha256": sha256,
        })

failed = [r for r in instance_rows if not r["structural_pass"]]

print("Generated:", len(instance_rows))
print("Structural PASS:", len(instance_rows) - len(failed))
print("Structural FAIL:", len(failed))

if failed:
    print("\nFailures:")
    for row in failed[:20]:
        print(row["instance_id"], "->", row["structural_errors"])

## 6. Validate the intended F1--F4 family treatments

**Final F4 interpretation.** The hard maintenance-incompatibility score is expected to follow `L < M = H`, not `L < M < H`. F4-H is stricter than F4-M through lower retained-capacity factors on the X1 and third protected activities.


In [ ]:
def check_f1(seed):
    errors = []
    L = cases[("F1_system_scale_L", seed)]
    M = cases[("F1_system_scale_M", seed)]
    H = cases[("F1_system_scale_H", seed)]

    if [len(L.assets), len(M.assets), len(H.assets)] != [8, 12, 16]:
        errors.append("F1 asset counts are not 8/12/16")
    if [len(L.maintenance_tasks), len(M.maintenance_tasks), len(H.maintenance_tasks)] != [8, 12, 16]:
        errors.append("F1 maintenance counts are not 8/12/16")
    if [len(L.arcs), len(M.arcs), len(H.arcs)] != [21, 33, 45]:
        errors.append("F1 arc counts are not 21/33/45")

    asset_sets = [{a.asset_id for a in s.assets} for s in (L, M, H)]
    if not (asset_sets[0] <= asset_sets[1] <= asset_sets[2]):
        errors.append("F1 asset sets are not nested")

    def arc_structures(s):
        return {
            (a.from_node, a.to_node, round(a.capacity_tph, 8), tuple(a.allowed_products))
            for a in s.arcs
        }

    arc_sets = [arc_structures(s) for s in (L, M, H)]
    if not (arc_sets[0] <= arc_sets[1] <= arc_sets[2]):
        errors.append("F1 arc structures are not nested")

    eff = [s.metadata.get("effective_network_capacity_tph") for s in (L, M, H)]
    if all(isinstance(x, (int, float)) for x in eff):
        if not (eff[0] < eff[1] < eff[2]):
            errors.append("F1 effective capacity does not increase L<M<H")

        supply_util = [
            mean_daily_total(s.inbound_supply) / (c * 24.0)
            for s, c in zip((L, M, H), eff)
        ]
        demand_util = [
            mean_daily_total(s.outbound_demand) / (c * 24.0)
            for s, c in zip((L, M, H), eff)
        ]
        if max(supply_util) - min(supply_util) > 1e-7:
            errors.append(f"F1 normalized supply differs across levels: {supply_util}")
        if max(demand_util) - min(demand_util) > 1e-7:
            errors.append(f"F1 normalized demand differs across levels: {demand_util}")

    return errors

def expected_f2_duration(full_duration, ratios):
    result = {}
    previous = 0
    for mode_id in gen.MAINTENANCE_MODE_ORDER:
        duration = math.ceil(full_duration * ratios[mode_id])
        duration = max(duration, previous + 1)
        result[mode_id] = duration
        previous = duration
    return result

def check_f2(seed):
    errors = []
    scenarios = {
        level: cases[(f"F2_mode_tradeoff_{level}", seed)]
        for level in "LMH"
    }

    if len({topology_signature(s) for s in scenarios.values()}) != 1:
        errors.append("F2 topology differs across L/M/H")
    if len({supply_signature(s) for s in scenarios.values()}) != 1:
        errors.append("F2 supply realization differs across L/M/H")
    if len({demand_signature(s) for s in scenarios.values()}) != 1:
        errors.append("F2 demand realization differs across L/M/H")
    if len({inventory_signature(s) for s in scenarios.values()}) != 1:
        errors.append("F2 initial inventory differs across L/M/H")
    if len({maintenance_core_signature(s) for s in scenarios.values()}) != 1:
        errors.append("F2 common maintenance realization/EM/LM differs across L/M/H")

    for level, scenario in scenarios.items():
        ratios = gen.F2_MODE_DURATION_RATIOS[level]
        for task in scenario.maintenance_tasks:
            modes = {m.mode_id: m.duration_days for m in task.modes}
            full = modes["full_shutdown"]
            expected = expected_f2_duration(full, ratios)
            if modes != expected:
                errors.append(
                    f"{level}/{task.task_id} mode durations {modes} != expected {expected}"
                )
                break

    return errors

def check_f3(seed):
    errors = []
    L = cases[("F3_yard_flow_L", seed)]
    M = cases[("F3_yard_flow_M", seed)]
    H = cases[("F3_yard_flow_H", seed)]

    # F3 must not change topology, supply/demand, or maintenance realization.
    if len({topology_signature(s) for s in (L, M, H)}) != 1:
        errors.append("F3 topology differs across L/M/H")
    if len({supply_signature(s) for s in (L, M, H)}) != 1:
        errors.append("F3 supply differs across L/M/H")
    if len({demand_signature(s) for s in (L, M, H)}) != 1:
        errors.append("F3 demand differs across L/M/H")
    if len({maintenance_core_signature(s) for s in (L, M, H)}) != 1:
        errors.append("F3 maintenance realization differs across L/M/H")

    zone_maps = [{z.zone_id: z for z in s.zones} for s in (L, M, H)]
    for zid in EXPECTED_ZONES:
        cap = [m[zid].capacity_tonnes for m in zone_maps]
        min_ratio = [m[zid].minimum_inventory_tonnes / m[zid].capacity_tonnes for m in zone_maps]
        if not (cap[0] >= cap[1] >= cap[2]):
            errors.append(f"F3 {zid} capacity is not L>=M>=H: {cap}")
        if not (min_ratio[0] <= min_ratio[1] <= min_ratio[2]):
            errors.append(f"F3 {zid} minimum ratio is not L<=M<=H: {min_ratio}")

    peak = [max(s.minimum_throughput_rate_tph.values()) for s in (L, M, H)]
    if not (peak[0] < peak[1] < peak[2]):
        errors.append(f"F3 throughput pressure is not L<M<H: {peak}")

    return errors

def conflict_score(scenario):
    score = 0
    for a in scenario.priority_activities:
        if a.maintenance_incompatible:
            protected_days = (
                a.duration_days
                + a.pre_campaign_buffer_days
                + a.post_campaign_buffer_days
            )
            score += protected_days * len(a.required_assets)
    return score

def check_f4(seed):
    errors = []
    L = cases[("F4_operational_conflict_L", seed)]
    M = cases[("F4_operational_conflict_M", seed)]
    H = cases[("F4_operational_conflict_H", seed)]

    levels = (L, M, H)

    # Final activity counts: L has two protected activities; M/H have three.
    if [len(s.priority_activities) for s in levels] != [2, 3, 3]:
        errors.append("F4 activity counts are not 2/3/3")

    def activities_by_id(scenario):
        return {a.activity_id: a for a in scenario.priority_activities}

    by_level = [activities_by_id(s) for s in levels]

    # Activity 1 is common and has the same retained-capacity factor at every level.
    event1 = [
        d.get("PA_EXPORT_WINDOW_1")
        for d in by_level
    ]
    if any(a is None for a in event1):
        errors.append("F4 PA_EXPORT_WINDOW_1 is missing from one or more levels")
    else:
        event1_factor = [a.capacity_factor_during_activity for a in event1]
        if len({round(v, 12) for v in event1_factor}) != 1:
            errors.append(
                f"F4 event-1 capacity factor should be common across L/M/H: "
                f"{event1_factor}"
            )

    # The X1 activity is present at every level and becomes strictly more severe L->M->H.
    x1 = [
        d.get("PA_X1_ROUTE_WINDOW")
        for d in by_level
    ]
    if any(a is None for a in x1):
        errors.append("F4 PA_X1_ROUTE_WINDOW is missing from one or more levels")
    else:
        x1_factor = [a.capacity_factor_during_activity for a in x1]
        if not (x1_factor[0] > x1_factor[1] > x1_factor[2]):
            errors.append(
                f"F4 X1 capacity severity is not L>M>H: {x1_factor}"
            )

    # Event 3 is omitted at L and is stricter at H than at M.
    event3_L = by_level[0].get("PA_EXPORT_WINDOW_3")
    event3_M = by_level[1].get("PA_EXPORT_WINDOW_3")
    event3_H = by_level[2].get("PA_EXPORT_WINDOW_3")

    if event3_L is not None:
        errors.append("F4 PA_EXPORT_WINDOW_3 should be omitted at L")
    if event3_M is None or event3_H is None:
        errors.append("F4 PA_EXPORT_WINDOW_3 should be present at M and H")
    else:
        event3_factor = [
            event3_M.capacity_factor_during_activity,
            event3_H.capacity_factor_during_activity,
        ]
        if not (event3_factor[0] > event3_factor[1]):
            errors.append(
                f"F4 event-3 capacity severity is not M>H: {event3_factor}"
            )

    # Final calibrated hard-incompatibility design:
    # L: none; M/H: X1 activity only. Hence exposure must satisfy L < M = H.
    incompatible_ids = [
        {
            a.activity_id
            for a in s.priority_activities
            if a.maintenance_incompatible
        }
        for s in levels
    ]
    expected_incompatible_ids = [
        set(),
        {"PA_X1_ROUTE_WINDOW"},
        {"PA_X1_ROUTE_WINDOW"},
    ]
    if incompatible_ids != expected_incompatible_ids:
        errors.append(
            f"F4 incompatible activity IDs differ from final design: "
            f"{incompatible_ids}"
        )

    scores = [conflict_score(s) for s in levels]
    if not (scores[0] < scores[1] and scores[1] == scores[2]):
        errors.append(
            f"F4 hard-incompatibility exposure is not L<M=H: {scores}"
        )

    # Shared activities use the same seeded timing across levels.
    shared = set(by_level[0]) & set(by_level[1]) & set(by_level[2])
    for activity_id in shared:
        timing = [
            (
                d[activity_id].start_day,
                d[activity_id].duration_days,
            )
            for d in by_level
        ]
        if len(set(timing)) != 1:
            errors.append(
                f"F4 shared activity timing differs for {activity_id}: {timing}"
            )

    return errors

family_checkers = {
    "F1": check_f1,
    "F2": check_f2,
    "F3": check_f3,
    "F4": check_f4,
}

family_rows = []
for seed in SEEDS:
    for family, checker in family_checkers.items():
        errors = checker(seed)
        family_rows.append({
            "family": family,
            "seed": seed,
            "pass": len(errors) == 0,
            "errors": " | ".join(errors),
        })

failed_family = [r for r in family_rows if not r["pass"]]

print("Family checks:", len(family_rows))
print("PASS:", len(family_rows) - len(failed_family))
print("FAIL:", len(failed_family))

if failed_family:
    print("\nFailures:")
    for row in failed_family:
        print(row["family"], "seed", row["seed"], "->", row["errors"])

## 7. Verify that the seeds actually change the realization

In [ ]:
def realization_fingerprint(scenario):
    payload = {
        "supply": supply_signature(scenario),
        "demand": demand_signature(scenario),
        "inventory": inventory_signature(scenario),
        "maintenance": maintenance_core_signature(scenario),
        "activities": tuple(
            (a.activity_id, a.start_day, a.end_day)
            for a in scenario.priority_activities
        ),
    }
    text = repr(payload).encode("utf-8")
    return hashlib.sha256(text).hexdigest()

diversity_rows = []
for name in experimental_names:
    fingerprints = {
        realization_fingerprint(cases[(name, seed)])
        for seed in SEEDS
    }
    diversity_rows.append({
        "scenario": name,
        "unique_realizations": len(fingerprints),
        "seed_count": len(SEEDS),
        "pass": len(fingerprints) == len(SEEDS),
    })

for row in diversity_rows:
    print(
        f"{row['scenario']}: "
        f"{row['unique_realizations']}/{row['seed_count']} unique "
        f"{'PASS' if row['pass'] else 'CHECK'}"
    )

## 8. Export validation reports and freeze manifest

In [ ]:
def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

instance_csv = OUTPUT_DIR / "benchmark_validation.csv"
family_csv = OUTPUT_DIR / "family_validation.csv"
diversity_csv = OUTPUT_DIR / "seed_diversity.csv"

write_csv(instance_csv, instance_rows)
write_csv(family_csv, family_rows)
write_csv(diversity_csv, diversity_rows)

manifest = {
    "generator_file": str(gen.__file__),
    "generator_randomization_version": getattr(
        gen, "GENERATOR_RANDOMIZATION_VERSION", "unknown"
    ),
    "rng_stream_version": getattr(gen, "RNG_STREAM_VERSION", "unknown"),
    "rng_stream_names": list(getattr(gen, "RNG_STREAM_NAMES", ())),
    "scenario_names": list(experimental_names),
    "seeds": list(SEEDS),
    "instance_count": len(instance_rows),
    "expected_instance_count": 120,
    "include_reference_schedule": INCLUDE_REFERENCE_SCHEDULE,
    "all_structural_checks_pass": all(r["structural_pass"] for r in instance_rows),
    "all_family_checks_pass": all(r["pass"] for r in family_rows),
    "all_seed_diversity_checks_pass": all(r["pass"] for r in diversity_rows),
    "files": [
        {
            "instance_id": r["instance_id"],
            "path": r["json_file"],
            "sha256": r["sha256"],
        }
        for r in instance_rows
        if r["json_file"]
    ],
}

manifest_path = OUTPUT_DIR / "benchmark_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")

print("Wrote:")
print(" -", instance_csv)
print(" -", family_csv)
print(" -", diversity_csv)
print(" -", manifest_path)

print("\nREADY TO FREEZE =", (
    manifest["instance_count"] == 120
    and manifest["all_structural_checks_pass"]
    and manifest["all_family_checks_pass"]
    and manifest["all_seed_diversity_checks_pass"]
))

## 9. Separate validation scenarios

`base_control` and `structurally_infeasible` are useful validation cases, but they are **not part of the 120-instance performance benchmark**.

The structural-infeasible case is only confirmed as infeasible after the exact MILP proves infeasibility.

In [ ]:
validation_cases = {}

for name in ("base_control", "structurally_infeasible"):
    s = gen.build_scenario(name, make_config(SEEDS[0]))
    validation_cases[name] = s
    errors = check_instance(s)
    print(
        name,
        "structural/data checks:",
        "PASS" if not errors else f"FAIL -> {errors}"
    )

## 10. Exact MILP feasibility hook

The generator's lightweight evaluator is **not** an exact model-feasibility proof because it does not enforce every network/stockyard constraint.

Wire your actual monolithic MILP feasibility routine here. The desired contract is:

```python
result = exact_feasibility_fn(
    model_data=scenario.optimizer_objects(),
    time_limit_sec=...,
    strict_capacity=False,
)
```

and `result` should contain at least:

```text
status: OPTIMAL | INFEASIBLE | TIME_LIMIT | ...
has_incumbent: True/False
```

Classification rule:

- `OPTIMAL` or any status with an incumbent -> **feasible**
- solver-proven `INFEASIBLE` -> **infeasible**
- time limit without incumbent -> **unresolved**, never "infeasible"

In [ ]:
RUN_EXACT_MILP = False

# Replace None with your actual callable after we connect this notebook to
# the current MILP implementation.
exact_feasibility_fn = None

def classify_feasibility(result):
    status = str(result.get("status", "")).upper()
    has_incumbent = bool(result.get("has_incumbent", False))

    if has_incumbent:
        return "FEASIBLE"
    if status == "OPTIMAL":
        return "FEASIBLE"
    if status == "INFEASIBLE":
        return "INFEASIBLE"
    return "UNRESOLVED"

if RUN_EXACT_MILP:
    assert callable(exact_feasibility_fn), (
        "Set exact_feasibility_fn to the monolithic MILP feasibility routine first."
    )

    feasibility_rows = []
    for (name, seed), scenario in cases.items():
        result = exact_feasibility_fn(
            model_data=scenario.optimizer_objects(),
            time_limit_sec=600,
            strict_capacity=False,
        )
        feasibility_rows.append({
            "scenario": name,
            "seed": seed,
            "classification": classify_feasibility(result),
            **result,
        })

    feasibility_csv = OUTPUT_DIR / "exact_milp_feasibility.csv"
    write_csv(feasibility_csv, feasibility_rows)
    print("Wrote:", feasibility_csv)
else:
    print(
        "Exact MILP validation is intentionally OFF. "
        "Connect exact_feasibility_fn before treating the dataset as solver-certified."
    )

## 11. Final acceptance rule

Freeze the dataset only when:

```text
120/120 instance structural checks pass
40/40 family-pairing checks pass
12/12 seed-diversity checks pass
and the exact MILP feasibility run has the intended classification (all 120 experimental cases feasible; the separate structural-infeasible control proven infeasible)
```

Do not replace individual seeds because one algorithm performs poorly on them.  
If a systematic feasibility problem appears, revise the predefined family calibration and regenerate the affected benchmark consistently.